In [183]:
import requests
import pandas as pd 
import numpy as np 
import pprint

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder 
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import cross_val_score

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier 

In [75]:
next_url = "https://pokeapi.co/api/v2/pokemon/"
pokemons = []
while (next_url):
    data = requests.get(next_url).json()
    next_url = data['next']
    pokemons.extend(data['results'])

print(len(pokemons))

1351


starting with the basic stats as interdependent variables for the ML model

In [130]:
pokemon_details = []
for poki in pokemons:
    poki_details = requests.get(poki['url']).json()
    name = poki['name']
    hp = poki_details['stats'][0]['base_stat']
    attack = poki_details['stats'][1]['base_stat']
    defense = poki_details['stats'][2]['base_stat']
    specialatt = poki_details['stats'][3]['base_stat']
    specialdef = poki_details['stats'][4]['base_stat']
    speed = poki_details['stats'][5]['base_stat']
    weight = poki_details['weight']
    pokitype = poki_details['types'][0]['type']['name']
    thispoki = {
        'name': name,
        'hp': hp,
        'attack': attack,
        'defense': defense,
        "specialatt": specialatt,
        "specialdef": specialdef,
        "speed": speed,
        "weight": weight,
        "pokitype": pokitype
    }
    pokemon_details.append(thispoki)

In [136]:
pokemon_df = pd.DataFrame(pokemon_details)
data_df = pokemon_df.copy()

In [167]:
y = data_df['pokitype']
X = data_df.drop(['pokitype', 'name'], axis=1)
print(X.columns)
print(X.head(10))

Index(['hp', 'attack', 'defense', 'specialatt', 'specialdef', 'speed',
       'weight'],
      dtype='str')
   hp  attack  defense  specialatt  specialdef  speed  weight
0  45      49       49          65          65     45      69
1  60      62       63          80          80     60     130
2  80      82       83         100         100     80    1000
3  39      52       43          60          50     65      85
4  58      64       58          80          65     80     190
5  78      84       78         109          85    100     905
6  44      48       65          50          64     43      90
7  59      63       80          65          80     58     225
8  79      83      100          85         105     78     855
9  45      30       35          20          20     45      29


In [169]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.2, random_state=42, stratify=y) 

X_num = Pipeline([
    ('Imputer', SimpleImputer(strategy="mean")),
    ("Scaler", StandardScaler())
])

X_cat = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder())
])

X_num_cols = X.columns
X_cat_cols = []

preprocessor = ColumnTransformer([
    ('preprocess_num', X_num, X_num_cols),
    ('preprocess_cat', X_cat, X_cat_cols)
])

In [172]:
baseline_LR = Pipeline([
    ('preprocessor', preprocessor),
    ('LR', LogisticRegression())
])

CV_LR = cross_val_score(
    baseline_LR, 
    X_train,
    y_train,
    scoring="roc_auc_ovr",
    cv=5,
)

CV_LR_score = np.mean(CV_LR)
print(CV_LR_score)

0.6682792428676809


In [176]:
baseline_DT = Pipeline([
    ('preprocessor', preprocessor),
    ('DT', DecisionTreeClassifier())
])

CV_DT = cross_val_score(
    baseline_DT,
    X_train,
    y_train,
    scoring="roc_auc_ovr",
    cv=5
)

CV_DT_score = np.mean(CV_DT)
print(CV_DT_score)

0.5796912411294717


In [179]:
baseline_RF = Pipeline([
    ('preprocessor', preprocessor),
    ('RF', RandomForestClassifier())
])

CV_RF = cross_val_score(
    baseline_RF, 
    X_train,
    y_train,
    cv=5,
    scoring="roc_auc_ovr" 
)

CV_RF_score = np.mean(CV_RF)
print(CV_RF_score)

0.747920215556847


In [182]:
baseline_GB = Pipeline([
    ('preprocessor', preprocessor),
    ('GB', GradientBoostingClassifier())
])

CV_GB = cross_val_score(
    baseline_GB,
    X_train, 
    y_train,
    cv=5,
    scoring="roc_auc_ovr"
)

CV_GB_score = np.mean(CV_GB)
print(CV_GB_score)

0.7212210915159486


In [185]:
baseline_KNN = Pipeline([
    ('preprocessor', preprocessor),
    ('KNN', KNeighborsClassifier())
])

CV_KNN = cross_val_score(
    baseline_KNN,
    X_train,
    y_train,
    cv=5,
    scoring="auc_roc_ovr"
)

CV_KNN_scoring = np.mean(CV_KNN)
print(CV_KNN_scoring)

InvalidParameterError: The 'scoring' parameter of cross_val_score must be a str among {'f1', 'average_precision', 'completeness_score', 'jaccard_micro', 'precision_weighted', 'd2_log_loss_score', 'jaccard_macro', 'homogeneity_score', 'top_k_accuracy', 'neg_median_absolute_error', 'accuracy', 'neg_mean_poisson_deviance', 'roc_auc_ovr', 'precision', 'positive_likelihood_ratio', 'recall_micro', 'neg_mean_gamma_deviance', 'neg_root_mean_squared_log_error', 'recall_weighted', 'neg_root_mean_squared_error', 'precision_samples', 'r2', 'd2_brier_score', 'recall_samples', 'matthews_corrcoef', 'neg_brier_score', 'rand_score', 'neg_log_loss', 'adjusted_rand_score', 'f1_micro', 'precision_micro', 'd2_absolute_error_score', 'f1_macro', 'neg_mean_absolute_percentage_error', 'adjusted_mutual_info_score', 'balanced_accuracy', 'jaccard', 'f1_weighted', 'f1_samples', 'fowlkes_mallows_score', 'mutual_info_score', 'jaccard_samples', 'roc_auc_ovo_weighted', 'normalized_mutual_info_score', 'roc_auc_ovr_weig